In [1]:
import pandas as pd
from datetime import timedelta

In [2]:
import ee

# 強制重新進行身份驗證流程（更換帳號一定要做）
# ee.Authenticate(force=True)
# 用以下的code才可以使用ee.batch.Export.image.toDrive()，存到Google Drive
ee.Authenticate(quiet=True)
ee.Initialize(project="floodmap-486207")

print("GEE 驗證成功！")

GEE 驗證成功！


In [ ]:
import pandas as pd
import ee
import re
from datetime import datetime, timedelta

# 2. 讀取資料
df = pd.read_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_final.csv', encoding='big5', encoding_errors='ignore')
selected_filename = [445,470,502]
df_select = df[df['filename'].isin(selected_filename)]

def process_imerg_v7(row):
    # --- 時間處理 ---
    end_date = pd.to_datetime(row['latest_timestamp'])
    start_date = end_date - timedelta(days=7)

    # --- 空間範圍 ---
    roi = ee.Geometry.Rectangle([row['west'], row['south'], row['east'], row['north']])

    # --- 抓取 IMERG V07 ---
    # 確保路徑完整：NASA/GPM_L3/IMERG_V07
    dataset = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
                .filterBounds(roi) \
                .filterDate(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')) \
                .select('precipitation')

    # --- 計算指標 ---
    # 累積降雨 (mm): 總和 * 0.5 (因為是半小時一筆)
    accumulated_precip = dataset.sum().multiply(0.5).clip(roi)
    # 最大降雨強度 (mm/hr)
    max_intensity = dataset.max().clip(roi)

    # --- 檔名與描述清理 (解決第 4 筆錯誤) ---
    # 取得名稱：有 event 用 event，沒內容就用 stac_item_id
    raw_name = str(row['event']) if pd.notna(row['event']) and str(row['event']).strip() != '' else str(row['stac_item_id'])

    # 強制取代所有非英數字元為底線，並限制長度在 50 字內
    clean_name = re.sub(r'[^a-zA-Z0-9]', '_', raw_name)
    clean_name = clean_name[:50]

    return accumulated_precip, max_intensity, roi, clean_name

# 3. 執行迴圈並啟動任務
for index, row in df_select.iterrows():
    try:
        acc_img, max_img, region, event_name = process_imerg_v7(row)

        # 檢查該時段是否有影像
        img_count = acc_img.bandNames().size().getInfo()
        if img_count == 0:
            print(f"⚠️ 第 {index} 筆 ({event_name}): V07 在此時段暫無資料。")
            continue

        # 匯出累積降雨
        task_acc = ee.batch.Export.image.toDrive(
            image=acc_img,
            description=f'Orig_V7_Acc_{event_name}',
            folder='IMERG_V7_Rainfall_orig',
            fileNamePrefix=f'Orig_Acc_{event_name}',
            region=region,
            scale=11132,
            fileFormat='GeoTIFF'
        )
        task_acc.start()

        # # 匯出最大強度
        # task_max = ee.batch.Export.image.toDrive(
        #     image=max_img,
        #     description=f'V7_Max_{event_name}',
        #     folder='IMERG_V7_Rainfall',
        #     fileNamePrefix=f'Max_{event_name}',
        #     region=region,
        #     scale=11132,
        #     fileFormat='GeoTIFF'
        # )
        # task_max.start()

        print(f"✅ 第 {index} 筆任務已啟動: {event_name}")

    except Exception as e:
        print(f"❌ 第 {index} 筆發生錯誤: {e}")

✅ 第 17 筆任務已啟動: Flood_in_Romania
✅ 第 18 筆任務已啟動: Flood_in_Togo
✅ 第 23 筆任務已啟動: Flood_in_Southern_Ireland


In [ ]:
import pandas as pd
import ee
import os
import re
from datetime import datetime, timedelta

# 2. 設定 Colab 路徑
csv_path = 'kurosiwo_final.csv' # 請確保已上傳檔案

# 3. 讀取資料
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, encoding='big5', encoding_errors='ignore')
    print(f"✅ 成功讀取 CSV")
else:
    print("❌ 檔案未上傳！請點擊左側檔案夾圖示並上傳 kurosiwo_final.csv")

def clean_name(name):
    if pd.isna(name) or str(name).strip() == '' or str(name).lower() == 'nan':
        return None
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', str(name))

# 4. 篩選出妳剛才「⚠️ 找不到」的那些事件名稱 (手動清單比對最準)
# 這裡列出妳剛才 log 裡面顯示失敗的關鍵字
failed_keywords = [
    "Sabaragamuwa", "Silute", "Thrace", "Ebro", "Aude",
    "Occitanie", "Vastra_Gotaland", "Landes", "Lazio",
    "Correze", "Queensland"
]

print("🚀 開始準備補抓任務...")

for index, row in df.iterrows():
    event_raw = str(row['event'])
    # 檢查這個事件是否在失敗清單中
    if any(k in event_raw for k in failed_keywords):
        event_name = clean_name(event_raw) or str(row['stac_item_id'])

        # GEE 運算邏輯
        roi = ee.Geometry.Rectangle([row['west'], row['south'], row['east'], row['north']])
        end_date = pd.to_datetime(row['latest_timestamp'])
        start_date = end_date - timedelta(days=7)

        # 抓取 V07
        dataset = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
                    .filterBounds(roi) \
                    .filterDate(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')) \
                    .select('precipitation')

        # 計算累積降雨
        acc_img = dataset.sum().multiply(0.5).clip(roi)

        # 匯出到 Google Drive (這最穩，因為匯出到本地 Colab 空間常會斷線)
        task = ee.batch.Export.image.toDrive(
            image=acc_img,
            description=f'REDO_V7_{event_name}'[:100],
            folder='IMERG_REDO_V7', # 妳的雲端硬碟會多這個資料夾
            fileNamePrefix=f'Acc_{event_name}',
            region=roi,
            scale=11132,
            fileFormat='GeoTIFF'
        )
        task.start()
        print(f"📡 任務已送出: {event_name}")

print("\n✅ 所有遺漏的任務已啟動！請到 Google Drive 查看進度。")

✅ 成功讀取 CSV
🚀 開始準備補抓任務...
📡 任務已送出: Floods_in_Sabaragamuwa_West-_Sri_Lanka
📡 任務已送出: Flood_in_Silute-_Lithuania
📡 任務已送出: Flood_in_Thrace-_Greece
📡 任務已送出: Flood_in_the_Ebro_river_basin-_Spain
📡 任務已送出: Floods_in_Aude-_France
📡 任務已送出: Flood_in_Occitanie-_France
📡 任務已送出: Flood_in_Landes-_France
📡 任務已送出: Flood_in_Lazio_Region-_Italy
📡 任務已送出: Flood_in_Correze_department-_France
📡 任務已送出: Floods_in_Queensland-_Australia
📡 任務已送出: Flood_in_the_Ebro_river_basin-_Spain
📡 任務已送出: Floods_in_Queensland-_Australia

✅ 所有遺漏的任務已啟動！請到 Google Drive 查看進度。
